<a href="https://colab.research.google.com/github/siddhesh1503/Applied-Data-Science/blob/main/ADS_KNN_Anomaly_SharkTank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anomaly / Outlier Detection — KNN Distance-Based (Unsupervised)
## ADS Experiment | Dataset: Shark Tank India (Seasons 1–5)
**Records:** 702 pitches | **Algorithm:** K-Nearest Neighbours | **Library:** PyOD + Scikit-learn

---

### Experiments in This Notebook

| # | Experiment | Rows | Features Used |
|---|-----------|------|---------------|
| A | **Pitch-Level Anomaly** | 702 | Ask Amount, Offered Equity, Valuation Requested, No. of Presenters, Season |
| B | **Deal-Level Anomaly** | 401 | Deal Amount, Deal Equity, Deal Valuation, No. of Sharks, Ask Amount |

---

> **Core Idea — KNN Anomaly Score:**
> For each startup pitch, compute the distance to its k-th nearest neighbour.
> Anomalous pitches are **isolated** (far from all others) so their k-NN distance is large.
> Normal pitches sit inside dense clusters → small k-NN distance.

---


## Cell 1 — Install & Import Libraries
---

In [ ]:
!pip install pyod scikit-learn matplotlib seaborn pandas numpy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors        import NearestNeighbors
from sklearn.preprocessing    import StandardScaler
from sklearn.decomposition    import PCA
from sklearn.metrics          import roc_auc_score, roc_curve, confusion_matrix
from pyod.models.knn          import KNN

# ── Dark plot theme ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0F1117',
    'axes.facecolor'   : '#1A1D27',
    'axes.edgecolor'   : '#3A3D4D',
    'axes.labelcolor'  : '#E0E0E0',
    'xtick.color'      : '#A0A0A0',
    'ytick.color'      : '#A0A0A0',
    'text.color'       : '#E0E0E0',
    'grid.color'       : '#2A2D3A',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.5,
    'legend.facecolor' : '#1A1D27',
    'legend.edgecolor' : '#3A3D4D',
    'font.family'      : 'DejaVu Sans',
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.titlecolor'  : '#FFFFFF',
})

NORMAL_COLOR  = '#2ECC71'
ANOMALY_COLOR = '#E74C3C'
SCORE_COLOR   = '#F39C12'
ACCENT        = '#3498DB'
W = 66

def hdr(title, char='='):
    print(char * W)
    print(f'  {title}')
    print(char * W)

def sep(char='-'):
    print(char * W)

def row(label, val=''):
    print(f'  {label:<30}{val}')

def blank():
    print()

print('Libraries imported. Dark theme ready.')


## Cell 2 — Load & Inspect Dataset
---

In [ ]:
# To fix this FileNotFoundError, please ensure the 'Shark Tank India.csv' file is accessible.
# You can:
# 1. Upload the file directly to your Colab environment by clicking the 'Files' icon on the left pane and then 'Upload'.
#    Once uploaded, the current code should work if the file is in the root directory.
# 2. Mount your Google Drive and specify the full path. For example:
#    from google.colab import drive
#    drive.mount('/content/drive')
#    df = pd.read_csv('/content/drive/MyDrive/YourFolderPath/Shark Tank India.csv')

df = pd.read_csv('Shark Tank India (1).csv')

hdr('SHARK TANK INDIA  —  Dataset Overview')
row('Total pitches       :', f'{len(df):,}')
row('Total columns       :', f'{df.shape[1]}')
row('Seasons covered     :', f'{df["Season Number"].nunique()}  (S1 to S5)')
row('Industries          :', f'{df["Industry"].nunique()}')
row('Unique startups     :', f'{df["Startup Name"].nunique()}')
row('Deals closed        :', f'{(df["Accepted Offer"].fillna(0)==1).sum()}  out of 702 pitches')
blank()
print('  Key Numeric Columns (completeness):')
sep()
key_cols = {
    'Original Ask Amount'   : '702 / 702  (100%)',
    'Original Offered Equity': '702 / 702  (100%)',
    'Valuation Requested'   : '702 / 702  (100%)',
    'Number of Presenters'  : '663 / 702  (94%)',
    'Yearly Revenue'        : '349 / 702  (50%)',
    'Total Deal Amount'     : '401 / 702  (57%)',
    'Total Deal Equity'     : '401 / 702  (57%)',
    'Deal Valuation'        : '400 / 702  (57%)',
    'Number of Sharks in Deal': '401 / 702  (57%)',
}
for k_, v_ in key_cols.items():
    row(f'  {k_}', v_)
blank()
df.head(3)

## Cell 3 — Feature Engineering & Dataset Preparation
---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EXPERIMENT A — All 702 Pitches  (Pitch-Level Anomaly Detection)
# Features: Ask Amount, Offered Equity, Valuation, Num Presenters, Season
# ══════════════════════════════════════════════════════════════════════════

hdr('EXPERIMENT A  —  Feature Engineering  (All 702 Pitches)')

df_a = df[[
    'Startup Name', 'Industry', 'Season Number',
    'Original Ask Amount',
    'Original Offered Equity',
    'Valuation Requested',
    'Number of Presenters',
]].copy()

# Derived features
df_a['Ask_per_Equity']      = df_a['Original Ask Amount'] / df_a['Original Offered Equity'].replace(0, np.nan)
df_a['Valuation_per_Lakh']  = df_a['Valuation Requested'] / df_a['Original Ask Amount'].replace(0, np.nan)

# Fill missing with median
df_a['Number of Presenters'].fillna(df_a['Number of Presenters'].median(), inplace=True)
df_a['Ask_per_Equity'].fillna(df_a['Ask_per_Equity'].median(), inplace=True)
df_a['Valuation_per_Lakh'].fillna(df_a['Valuation_per_Lakh'].median(), inplace=True)

FEAT_A = [
    'Original Ask Amount',
    'Original Offered Equity',
    'Valuation Requested',
    'Number of Presenters',
    'Ask_per_Equity',
    'Valuation_per_Lakh',
]
X_a = df_a[FEAT_A].values

blank()
print('  Features used in Experiment A:')
for i, f in enumerate(FEAT_A, 1):
    print(f'    [{i}] {f}')
blank()
print(f'  Shape of feature matrix : {X_a.shape}')
print(f'  Missing values after fill: {np.isnan(X_a).sum()}')
blank()

# ══════════════════════════════════════════════════════════════════════════
# EXPERIMENT B — 401 Closed Deals  (Deal-Level Anomaly Detection)
# ══════════════════════════════════════════════════════════════════════════
hdr('EXPERIMENT B  —  Feature Engineering  (401 Closed Deals)')

df_b = df[df['Accepted Offer'] == 1][[
    'Startup Name', 'Industry', 'Season Number',
    'Original Ask Amount',
    'Total Deal Amount',
    'Total Deal Equity',
    'Deal Valuation',
    'Number of Sharks in Deal',
]].dropna(subset=['Total Deal Amount','Total Deal Equity','Deal Valuation']).copy()

# Derived features
df_b['Deal_vs_Ask_Ratio']   = df_b['Total Deal Amount'] / df_b['Original Ask Amount'].replace(0, np.nan)
df_b['Equity_per_Lakh']     = df_b['Total Deal Equity'] / df_b['Total Deal Amount'].replace(0, np.nan)
df_b['Deal_vs_Ask_Ratio'].fillna(df_b['Deal_vs_Ask_Ratio'].median(), inplace=True)
df_b['Equity_per_Lakh'].fillna(df_b['Equity_per_Lakh'].median(), inplace=True)

FEAT_B = [
    'Original Ask Amount',
    'Total Deal Amount',
    'Total Deal Equity',
    'Deal Valuation',
    'Number of Sharks in Deal',
    'Deal_vs_Ask_Ratio',
    'Equity_per_Lakh',
]
X_b = df_b[FEAT_B].values

blank()
print('  Features used in Experiment B:')
for i, f in enumerate(FEAT_B, 1):
    print(f'    [{i}] {f}')
blank()
print(f'  Shape of feature matrix : {X_b.shape}')
print(f'  Missing values after fill: {np.isnan(X_b).sum()}')
blank()


## Cell 4 — Theory: KNN Distance-Based Anomaly Detection
---

In [ ]:
hdr('THEORY  —  KNN Distance-Based Anomaly Detection')
blank()
print('  HOW IT WORKS')
sep()
print('  Step 1  Standardise features (equal weight to all dimensions)')
print('  Step 2  Choose k (number of neighbours to consider)')
print('  Step 3  For each point x_i, find its k nearest neighbours')
print('          using Euclidean distance in the feature space')
print('  Step 4  Compute anomaly score = distance to the k-th neighbour')
print('          (large distance = isolated = potential anomaly)')
print('  Step 5  Apply threshold based on contamination rate')
print('  Step 6  Flag points above threshold as ANOMALY')
blank()
print('  ON SHARK TANK DATA')
sep()
print('  A "normal" pitch  ->  ask amount, equity, valuation are typical')
print('                        similar to many other pitches  ->  low score')
print('  An anomaly pitch  ->  unusual combination of features:')
print('                        e.g., Rs 30,000L ask with 0.5% equity offered')
print('                        very different from all others  ->  high score')
blank()
print('  THREE SCORING METHODS (PyOD KNN)')
sep()
print('  largest  :  score = distance to the k-th nearest neighbour only')
print('  mean     :  score = average distance to all k neighbours')
print('  median   :  score = median distance to all k neighbours')
blank()
print('  THRESHOLD STRATEGY')
sep()
print('  contamination = expected % of anomalies in data')
print('  threshold = percentile(scores, (1 - contamination) * 100)')
print('  Points above threshold -> flagged as ANOMALY')
blank()
print('  KEY HYPERPARAMETER: k')
sep()
print('  Small k (2-5)   ->  sensitive to local noise')
print('  Large k (20-50) ->  smoother, captures global structure')
print('  Rule of thumb   ->  k = sqrt(n) as starting point')
blank()


## Cell 5 — Experiment A: Manual Step-by-Step (All 702 Pitches)
---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EXPERIMENT A  —  Manual KNN Anomaly Detection (All 9 steps)
# ══════════════════════════════════════════════════════════════════════════

# ─── STEP 1: Standardise ─────────────────────────────────────────────────
hdr('STEP 1   STANDARDISE FEATURES')
scaler_a = StandardScaler()
X_a_scaled = scaler_a.fit_transform(X_a)

print('  Formula  :  z = (x - mean) / std')
blank()
for i, f in enumerate(FEAT_A):
    col = X_a[:, i]
    print(f'  {f:<32}  mean={col.mean():>9.2f}  std={col.std():>9.2f}')
blank()
print(f'  Before scaling  ->  value range: [{X_a.min():.2f}, {X_a.max():.2f}]')
print(f'  After  scaling  ->  value range: [{X_a_scaled.min():.2f}, {X_a_scaled.max():.2f}]')
blank()

# ─── STEP 2: Choose k ────────────────────────────────────────────────────
hdr('STEP 2   CHOOSE k  (Number of Neighbours)')
n_a = len(X_a_scaled)
k_a = int(np.sqrt(n_a))
print(f'  Total pitches n          = {n_a}')
print(f'  Rule of thumb k=sqrt(n)  = sqrt({n_a}) = {np.sqrt(n_a):.2f}  ->  k = {k_a}')
blank()

# ─── STEP 3: Fit KNN & Get Distances ─────────────────────────────────────
hdr('STEP 3   FIT KNN MODEL & COMPUTE DISTANCES')
knn_a = NearestNeighbors(n_neighbors=k_a+1, algorithm='ball_tree', metric='euclidean')
knn_a.fit(X_a_scaled)
distances_a, indices_a = knn_a.kneighbors(X_a_scaled)
knn_scores_a = distances_a[:, k_a]   # distance to k-th neighbour (exclude self)

print(f'  NearestNeighbors(n_neighbors={k_a+1}, metric="euclidean", algorithm="ball_tree")')
print(f'  distances matrix shape : {distances_a.shape}')
blank()
print(f'  Top 10 pitches by HIGHEST anomaly score (most unusual pitches):')
sep()
top_idx = np.argsort(knn_scores_a)[::-1][:10]
print(f'  {"Rank":<5} {"Startup Name":<30} {"Industry":<25} {"KNN Score":>10}')
sep()
for rank, idx in enumerate(top_idx, 1):
    name     = df_a['Startup Name'].iloc[idx][:28]
    industry = df_a['Industry'].iloc[idx][:23]
    score    = knn_scores_a[idx]
    print(f'  {rank:<5} {name:<30} {industry:<25} {score:>10.4f}')
blank()

# ─── STEP 4: Anomaly Score Stats ─────────────────────────────────────────
hdr('STEP 4   ANOMALY SCORE DISTRIBUTION')
print(f'  Min score    : {knn_scores_a.min():.4f}')
print(f'  Max score    : {knn_scores_a.max():.4f}')
print(f'  Mean score   : {knn_scores_a.mean():.4f}')
print(f'  Std score    : {knn_scores_a.std():.4f}')
print(f'  Median score : {np.median(knn_scores_a):.4f}')
print(f'  75th pctile  : {np.percentile(knn_scores_a, 75):.4f}')
print(f'  90th pctile  : {np.percentile(knn_scores_a, 90):.4f}')
print(f'  95th pctile  : {np.percentile(knn_scores_a, 95):.4f}')
blank()

# ─── STEP 5: Set Threshold ────────────────────────────────────────────────
hdr('STEP 5   SET ANOMALY THRESHOLD')
contamination_a = 0.05   # expect ~5% of pitches to be outliers
threshold_a = np.percentile(knn_scores_a, (1 - contamination_a) * 100)
n_flagged_a = (knn_scores_a > threshold_a).sum()
print(f'  Contamination assumed    : {contamination_a*100:.0f}%')
print(f'  Percentile threshold     : {(1-contamination_a)*100:.0f}th  ->  {threshold_a:.4f}')
print(f'  Points flagged as anomaly: {n_flagged_a}  out of {n_a}  ({n_flagged_a/n_a*100:.1f}%)')
blank()

# ─── STEP 6: Classify ─────────────────────────────────────────────────────
hdr('STEP 6   CLASSIFY POINTS')
y_pred_a_manual = (knn_scores_a > threshold_a).astype(int)
print(f'  Normal pitches  : {(y_pred_a_manual==0).sum()}')
print(f'  Anomaly pitches : {(y_pred_a_manual==1).sum()}')
blank()


## Cell 6 — Experiment A: PyOD KNN (All 3 Methods)
---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Experiment A — PyOD KNN: largest / mean / median
# ══════════════════════════════════════════════════════════════════════════

methods_a = ['largest', 'mean', 'median']
res_a = {}

hdr('EXPERIMENT A  —  PyOD KNN (Three Scoring Methods)')
print(f'  {"Method":<12} {"Anomalies":>10} {"Min Score":>10} {"Max Score":>10} {"Mean Score":>12}')
sep()

for method in methods_a:
    clf = KNN(n_neighbors=k_a, method=method, contamination=contamination_a)
    clf.fit(X_a_scaled)
    res_a[method] = {
        'labels' : clf.labels_,
        'scores' : clf.decision_scores_,
        'clf'    : clf,
    }
    lbl = clf.labels_
    sc  = clf.decision_scores_
    print(f'  {method:<12} {lbl.sum():>10} {sc.min():>10.4f} {sc.max():>10.4f} {sc.mean():>12.4f}')

blank()
# Use 'largest' as primary
labels_a  = res_a['largest']['labels']
scores_a  = res_a['largest']['scores']

print(f'  Primary method   :  largest  (k-th NN distance)')
print(f'  Anomalies found  :  {labels_a.sum()}  pitches flagged')
blank()

# Attach results to dataframe
df_a['anomaly_score'] = scores_a
df_a['is_anomaly']    = labels_a

hdr('TOP 15 ANOMALOUS PITCHES  —  Experiment A (Pitch Level)')
sep()
top15_a = df_a[df_a['is_anomaly']==1].nlargest(15, 'anomaly_score')
print(f'  {"#":<4} {"Startup Name":<28} {"Industry":<22} {"Ask (L)":>8} {"Equity%":>8} {"Valuation":>10} {"Score":>8}')
sep()
for rank, (_, row_) in enumerate(top15_a.iterrows(), 1):
    print(f'  {rank:<4} {str(row_["Startup Name"])[:26]:<28} '
          f'{str(row_["Industry"])[:20]:<22} '
          f'{row_["Original Ask Amount"]:>8.0f} '
          f'{row_["Original Offered Equity"]:>8.2f} '
          f'{row_["Valuation Requested"]:>10.0f} '
          f'{row_["anomaly_score"]:>8.4f}')
blank()


## Cell 7 — Visualization: Experiment A (6-Panel Dashboard)
---

In [ ]:
fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('Experiment A — KNN Anomaly Detection on All 702 Pitches\n'
             f'Features: Ask Amount, Equity, Valuation, Presenters, Ask/Equity, Valuation Ratio   |   k = {k_a}   |   contamination = {contamination_a*100:.0f}%',
             fontsize=15, fontweight='bold', color='white', y=1.01)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.34)
ax1, ax2, ax3 = fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]), fig.add_subplot(gs[0,2])
ax4, ax5, ax6 = fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1]), fig.add_subplot(gs[1,2])

for ax in [ax1,ax2,ax3,ax4,ax5,ax6]:
    ax.set_facecolor('#1A1D27')

# ── Panel 1: Ask Amount vs Offered Equity (coloured by anomaly) ────────────
norm_mask = labels_a == 0
anom_mask = labels_a == 1
ax1.scatter(df_a.loc[norm_mask,'Original Ask Amount'],
            df_a.loc[norm_mask,'Original Offered Equity'],
            c=NORMAL_COLOR, s=25, alpha=0.60, label=f'Normal ({norm_mask.sum()})', zorder=3)
ax1.scatter(df_a.loc[anom_mask,'Original Ask Amount'],
            df_a.loc[anom_mask,'Original Offered Equity'],
            c=ANOMALY_COLOR, s=120, marker='X', alpha=0.95,
            edgecolors='white', linewidth=0.7,
            label=f'Anomaly ({anom_mask.sum()})', zorder=4)
ax1.set_xlabel('Ask Amount (Rs Lakhs)', fontsize=10)
ax1.set_ylabel('Offered Equity (%)', fontsize=10)
ax1.set_title('Ask Amount vs Offered Equity\nRed X = Anomalous Pitches', fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_xlim(left=0)

# ── Panel 2: Ask Amount vs Valuation Requested (score heatmap) ────────────
sc_plot = ax2.scatter(df_a['Original Ask Amount'],
                       df_a['Valuation Requested'],
                       c=scores_a, cmap='RdYlGn_r',
                       s=35, alpha=0.85, edgecolors='none')
cbar1 = plt.colorbar(sc_plot, ax=ax2)
cbar1.set_label('Anomaly Score', color='white', fontsize=9)
cbar1.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar1.ax.yaxis.get_ticklabels(), color='white')
ax2.set_xlabel('Ask Amount (Rs Lakhs)', fontsize=10)
ax2.set_ylabel('Valuation Requested (Rs Lakhs)', fontsize=10)
ax2.set_title('Ask Amount vs Valuation\nScore Heatmap (Red = High Anomaly Score)', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(left=0); ax2.set_ylim(bottom=0)

# ── Panel 3: Anomaly Score Distribution ────────────────────────────────────
ax3.hist(scores_a[labels_a==0], bins=40, color=NORMAL_COLOR,
         alpha=0.75, label='Normal pitches', edgecolor='none', density=True)
ax3.hist(scores_a[labels_a==1], bins=15, color=ANOMALY_COLOR,
         alpha=0.85, label='Anomaly pitches', edgecolor='none', density=True)
thresh_val_a = np.percentile(scores_a, (1-contamination_a)*100)
ax3.axvline(thresh_val_a, color=SCORE_COLOR, linewidth=2.5, linestyle='--',
            label=f'Threshold = {thresh_val_a:.3f}')
ax3.set_xlabel('Anomaly Score (k-NN Distance)', fontsize=10)
ax3.set_ylabel('Density', fontsize=10)
ax3.set_title('Score Distribution\nSeparation between Normal & Anomaly', fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

# ── Panel 4: PCA 2D projection ─────────────────────────────────────────────
pca_a = PCA(n_components=2, random_state=42)
X_a_2d = pca_a.fit_transform(X_a_scaled)
ax4.scatter(X_a_2d[norm_mask,0], X_a_2d[norm_mask,1],
            c=NORMAL_COLOR, s=20, alpha=0.55, label='Normal', zorder=3)
ax4.scatter(X_a_2d[anom_mask,0], X_a_2d[anom_mask,1],
            c=ANOMALY_COLOR, s=130, marker='X', alpha=0.95,
            edgecolors='white', linewidth=0.7, label='Anomaly', zorder=4)
for idx in np.where(anom_mask)[0][:5]:
    name = df_a['Startup Name'].iloc[idx][:12]
    ax4.annotate(name,
                 xy=(X_a_2d[idx,0], X_a_2d[idx,1]),
                 xytext=(X_a_2d[idx,0]+0.3, X_a_2d[idx,1]+0.3),
                 fontsize=6.5, color='#FFAAAA',
                 arrowprops=dict(arrowstyle='->', color='#FFAAAA', lw=1.0))
ax4.set_xlabel(f'PC1  ({pca_a.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=10)
ax4.set_ylabel(f'PC2  ({pca_a.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=10)
ax4.set_title(f'PCA 2D Projection\n(Top 5 anomalies labelled)', fontweight='bold')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

# ── Panel 5: Anomalies by Industry ────────────────────────────────────────
ind_anom = df_a[df_a['is_anomaly']==1]['Industry'].value_counts()
ind_norm = df_a[df_a['is_anomaly']==0]['Industry'].value_counts()
# Top industries with anomalies
top_inds = ind_anom.nlargest(10)
colors_bar = [ANOMALY_COLOR]*len(top_inds)
bars5 = ax5.barh(range(len(top_inds)), top_inds.values,
                  color=colors_bar, edgecolor='none', alpha=0.85, height=0.65)
for i, (bar_, val) in enumerate(zip(bars5, top_inds.values)):
    ax5.text(val + 0.05, i, str(val),
             va='center', fontsize=9, fontweight='bold', color='white')
ax5.set_yticks(range(len(top_inds)))
ax5.set_yticklabels(top_inds.index, fontsize=8.5)
ax5.set_xlabel('Number of Anomalies', fontsize=10)
ax5.set_title('Anomaly Count by Industry\n(Which industries have unusual pitches?)', fontweight='bold')
ax5.grid(True, axis='x', alpha=0.3)
ax5.set_xlim(0, top_inds.max() * 1.3)

# ── Panel 6: Score by Season ───────────────────────────────────────────────
season_scores = [scores_a[df_a['Season Number'].values == s] for s in [1,2,3,4,5]]
bp = ax6.boxplot(season_scores, patch_artist=True, widths=0.55,
                 medianprops=dict(color='white', linewidth=2.5),
                 flierprops=dict(marker='X', markerfacecolor=ANOMALY_COLOR,
                                 markersize=8, alpha=0.85, markeredgecolor='none'))
season_colors = ['#2ECC71','#3498DB','#9B59B6','#F39C12','#E74C3C']
for patch, color in zip(bp['boxes'], season_colors):
    patch.set_facecolor(color + '88')
    patch.set_edgecolor(color)
ax6.set_xticks(range(1,6))
ax6.set_xticklabels([f'Season {s}' for s in range(1,6)], fontsize=9)
ax6.set_ylabel('Anomaly Score', fontsize=10)
ax6.set_title('Anomaly Score Distribution by Season\n(Outlier X marks = anomalous pitches)', fontweight='bold')
ax6.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Experiment A complete.  Anomalies detected: {labels_a.sum()}  out of {len(labels_a)} pitches.')


## Cell 8 — Experiment B: Manual Step-by-Step (401 Closed Deals)
---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EXPERIMENT B  —  Deal-Level Anomaly Detection (401 rows)
# ══════════════════════════════════════════════════════════════════════════

hdr('STEP 1   STANDARDISE FEATURES  (Deal Level)')
scaler_b = StandardScaler()
X_b_scaled = scaler_b.fit_transform(X_b)
blank()
for i, f in enumerate(FEAT_B):
    col = X_b[:, i]
    print(f'  {f:<32}  mean={col.mean():>9.2f}  std={col.std():>9.2f}')
blank()

hdr('STEP 2   CHOOSE k')
n_b = len(X_b_scaled)
k_b = int(np.sqrt(n_b))
print(f'  Total deals n          = {n_b}')
print(f'  k = sqrt({n_b}) = {np.sqrt(n_b):.2f}  ->  k = {k_b}')
blank()

hdr('STEP 3   FIT KNN & COMPUTE DISTANCES')
knn_b = NearestNeighbors(n_neighbors=k_b+1, algorithm='ball_tree', metric='euclidean')
knn_b.fit(X_b_scaled)
distances_b, _ = knn_b.kneighbors(X_b_scaled)
knn_scores_b   = distances_b[:, k_b]

print(f'  NearestNeighbors(n_neighbors={k_b+1}, metric="euclidean")')
print(f'  Distances matrix shape : {distances_b.shape}')
blank()

hdr('STEP 4   ANOMALY SCORE STATS')
print(f'  Min score    : {knn_scores_b.min():.4f}')
print(f'  Max score    : {knn_scores_b.max():.4f}')
print(f'  Mean score   : {knn_scores_b.mean():.4f}')
print(f'  Std score    : {knn_scores_b.std():.4f}')
print(f'  95th pctile  : {np.percentile(knn_scores_b, 95):.4f}')
blank()

hdr('STEP 5   SET THRESHOLD')
contamination_b = 0.05
threshold_b = np.percentile(knn_scores_b, (1 - contamination_b) * 100)
print(f'  Contamination          : {contamination_b*100:.0f}%')
print(f'  Threshold (95th pctile): {threshold_b:.4f}')
print(f'  Deals flagged          : {(knn_scores_b > threshold_b).sum()}')
blank()

hdr('STEP 6   PyOD KNN  (Three Methods)')
methods_b = ['largest','mean','median']
res_b = {}
print(f'  {"Method":<12} {"Anomalies":>10} {"Max Score":>10} {"Mean Score":>12}')
sep()
for method in methods_b:
    clf_b = KNN(n_neighbors=k_b, method=method, contamination=contamination_b)
    clf_b.fit(X_b_scaled)
    res_b[method] = {'labels': clf_b.labels_, 'scores': clf_b.decision_scores_}
    print(f'  {method:<12} {clf_b.labels_.sum():>10} {clf_b.decision_scores_.max():>10.4f} {clf_b.decision_scores_.mean():>12.4f}')

blank()
labels_b  = res_b['largest']['labels']
scores_b  = res_b['largest']['scores']
df_b['anomaly_score'] = scores_b
df_b['is_anomaly']    = labels_b

hdr('TOP 15 ANOMALOUS DEALS  —  Experiment B (Deal Level)')
sep()
top15_b = df_b[df_b['is_anomaly']==1].nlargest(15, 'anomaly_score')
print(f'  {"#":<4} {"Startup Name":<28} {"Deal Amt":>9} {"Deal Eq%":>9} {"Valuation":>10} {"Sharks":>7} {"Score":>8}')
sep()
for rank, (_, row_) in enumerate(top15_b.iterrows(), 1):
    print(f'  {rank:<4} {str(row_["Startup Name"])[:26]:<28} '
          f'{row_["Total Deal Amount"]:>9.0f} '
          f'{row_["Total Deal Equity"]:>9.2f} '
          f'{row_["Deal Valuation"]:>10.0f} '
          f'{row_["Number of Sharks in Deal"]:>7.0f} '
          f'{row_["anomaly_score"]:>8.4f}')
blank()


## Cell 9 — Visualization: Experiment B (Deal-Level)
---

In [ ]:
fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('Experiment B — KNN Anomaly Detection on 401 Closed Deals\n'
             f'Features: Deal Amount, Equity, Valuation, Sharks, Deal/Ask Ratio   |   k = {k_b}   |   contamination = {contamination_b*100:.0f}%',
             fontsize=15, fontweight='bold', color='white', y=1.01)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.34)
ax1, ax2, ax3 = fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]), fig.add_subplot(gs[0,2])
ax4, ax5, ax6 = fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1]), fig.add_subplot(gs[1,2])

for ax in [ax1,ax2,ax3,ax4,ax5,ax6]:
    ax.set_facecolor('#1A1D27')

b_norm = labels_b == 0
b_anom = labels_b == 1

# ── Panel 1: Deal Amount vs Deal Equity ────────────────────────────────────
ax1.scatter(df_b.loc[b_norm,'Total Deal Amount'],
            df_b.loc[b_norm,'Total Deal Equity'],
            c=NORMAL_COLOR, s=30, alpha=0.65, label=f'Normal ({b_norm.sum()})', zorder=3)
ax1.scatter(df_b.loc[b_anom,'Total Deal Amount'],
            df_b.loc[b_anom,'Total Deal Equity'],
            c=ANOMALY_COLOR, s=150, marker='X', alpha=0.95,
            edgecolors='white', linewidth=0.7,
            label=f'Anomaly ({b_anom.sum()})', zorder=4)
ax1.set_xlabel('Total Deal Amount (Rs Lakhs)', fontsize=10)
ax1.set_ylabel('Total Deal Equity (%)', fontsize=10)
ax1.set_title('Deal Amount vs Deal Equity\nRed X = Anomalous Deals', fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# ── Panel 2: Deal Amount vs Deal Valuation (score heatmap) ─────────────────
sc2 = ax2.scatter(df_b['Total Deal Amount'],
                   df_b['Deal Valuation'],
                   c=scores_b, cmap='RdYlGn_r',
                   s=45, alpha=0.88, edgecolors='none')
cbar2 = plt.colorbar(sc2, ax=ax2)
cbar2.set_label('Anomaly Score', color='white', fontsize=9)
cbar2.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color='white')
ax2.set_xlabel('Total Deal Amount (Rs Lakhs)', fontsize=10)
ax2.set_ylabel('Deal Valuation (Rs Lakhs)', fontsize=10)
ax2.set_title('Deal Amount vs Valuation\nScore Heatmap', fontweight='bold')
ax2.grid(True, alpha=0.3)

# ── Panel 3: Score Distribution ────────────────────────────────────────────
ax3.hist(scores_b[labels_b==0], bins=35, color=NORMAL_COLOR,
         alpha=0.75, label='Normal deals', edgecolor='none', density=True)
ax3.hist(scores_b[labels_b==1], bins=12, color=ANOMALY_COLOR,
         alpha=0.85, label='Anomaly deals', edgecolor='none', density=True)
thr_b_viz = np.percentile(scores_b, (1-contamination_b)*100)
ax3.axvline(thr_b_viz, color=SCORE_COLOR, linewidth=2.5, linestyle='--',
            label=f'Threshold = {thr_b_viz:.3f}')
ax3.set_xlabel('Anomaly Score (k-NN Distance)', fontsize=10)
ax3.set_ylabel('Density', fontsize=10)
ax3.set_title('Score Distribution — Deals\nNormal vs Anomaly Separation', fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

# ── Panel 4: PCA 2D projection ─────────────────────────────────────────────
pca_b = PCA(n_components=2, random_state=42)
X_b_2d = pca_b.fit_transform(X_b_scaled)
ax4.scatter(X_b_2d[b_norm,0], X_b_2d[b_norm,1],
            c=NORMAL_COLOR, s=25, alpha=0.55, label='Normal deal', zorder=3)
ax4.scatter(X_b_2d[b_anom,0], X_b_2d[b_anom,1],
            c=ANOMALY_COLOR, s=150, marker='X', alpha=0.95,
            edgecolors='white', linewidth=0.7, label='Anomaly deal', zorder=4)
for idx in np.where(b_anom)[0][:5]:
    name = df_b['Startup Name'].iloc[idx][:12]
    ax4.annotate(name,
                 xy=(X_b_2d[idx,0], X_b_2d[idx,1]),
                 xytext=(X_b_2d[idx,0]+0.3, X_b_2d[idx,1]+0.3),
                 fontsize=6.5, color='#FFAAAA',
                 arrowprops=dict(arrowstyle='->', color='#FFAAAA', lw=1.0))
ax4.set_xlabel(f'PC1  ({pca_b.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=10)
ax4.set_ylabel(f'PC2  ({pca_b.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=10)
ax4.set_title('PCA 2D Projection — Deals\n(Top 5 anomalies labelled)', fontweight='bold')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

# ── Panel 5: Deal Amount comparison — Normal vs Anomaly ────────────────────
ax5.boxplot([df_b[b_norm]['Total Deal Amount'].clip(upper=300),
             df_b[b_anom]['Total Deal Amount']],
            patch_artist=True, widths=0.55,
            medianprops=dict(color='white', linewidth=2.5),
            flierprops=dict(marker='X', markersize=7, alpha=0.8,
                            markerfacecolor=ANOMALY_COLOR, markeredgecolor='none'))
for patch, color in zip(ax5.patches, [NORMAL_COLOR, ANOMALY_COLOR]):
    patch.set_facecolor(color + '88')
    patch.set_edgecolor(color)
ax5.set_xticks([1,2]); ax5.set_xticklabels(['Normal Deals','Anomaly Deals'], fontsize=10)
ax5.set_ylabel('Total Deal Amount (Rs Lakhs)', fontsize=10)
ax5.set_title('Deal Amount: Normal vs Anomaly\n(Anomalies tend to be extreme values)', fontweight='bold')
ax5.grid(True, axis='y', alpha=0.3)
mu_n = df_b[b_norm]['Total Deal Amount'].mean()
mu_a = df_b[b_anom]['Total Deal Amount'].mean()
ax5.text(1, mu_n + 5, f'Mean: Rs {mu_n:.0f}L', ha='center', fontsize=9, color=NORMAL_COLOR, fontweight='bold')
ax5.text(2, mu_a + 5, f'Mean: Rs {mu_a:.0f}L', ha='center', fontsize=9, color=ANOMALY_COLOR, fontweight='bold')

# ── Panel 6: k sensitivity ─────────────────────────────────────────────────
k_vals_b = [3, 5, 10, 15, 20, k_b, 30]
k_vals_b = sorted(list(set(k_vals_b)))
n_anomalies_k = []
for kv in k_vals_b:
    clf_tmp = KNN(n_neighbors=kv, method='largest', contamination=contamination_b)
    clf_tmp.fit(X_b_scaled)
    n_anomalies_k.append(clf_tmp.labels_.sum())

ax6.plot(k_vals_b, n_anomalies_k, 'o-', color=SCORE_COLOR,
         linewidth=2.5, markersize=9, markerfacecolor='white',
         markeredgecolor=SCORE_COLOR, markeredgewidth=2.5)
ax6.axvline(k_b, color=ACCENT, linewidth=1.8, linestyle='--', alpha=0.8,
            label=f'Chosen k = {k_b}  (sqrt(n))')
for kv, na in zip(k_vals_b, n_anomalies_k):
    ax6.text(kv, na+0.3, str(na), ha='center', fontsize=9,
             fontweight='bold', color='white')
ax6.set_xlabel('k  (Number of Neighbours)', fontsize=10)
ax6.set_ylabel('Anomalies Detected', fontsize=10)
ax6.set_title('k Sensitivity — Deals\nEffect of k on Number of Anomalies', fontweight='bold')
ax6.legend(fontsize=9); ax6.grid(True, alpha=0.3)
ax6.set_ylim(0, max(n_anomalies_k) * 1.35)

plt.tight_layout()
plt.show()
print(f'Experiment B complete.  Anomalies detected: {labels_b.sum()}  out of {len(labels_b)} closed deals.')


## Cell 10 — k Sensitivity Analysis: Experiment A
---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# How does choice of k affect the anomaly scores and flagged startups?
# ══════════════════════════════════════════════════════════════════════════

k_values_a = [3, 5, 10, k_a, 30, 50]
k_values_a = sorted(list(set(k_values_a)))
k_results_a = {}

hdr('k SENSITIVITY ANALYSIS  —  Experiment A  (All 702 Pitches)')
print(f'  {"k":<8} {"Anomalies":>10} {"Avg Score (all)":>16} {"Avg Score (anom)":>17} {"Max Score":>10}')
sep()

for kv in k_values_a:
    clf_k = KNN(n_neighbors=kv, method='largest', contamination=contamination_a)
    clf_k.fit(X_a_scaled)
    lbl_k = clf_k.labels_
    sc_k  = clf_k.decision_scores_
    k_results_a[kv] = {'labels': lbl_k, 'scores': sc_k}
    avg_all  = sc_k.mean()
    avg_anom = sc_k[lbl_k==1].mean() if lbl_k.sum()>0 else 0
    print(f'  {kv:<8} {lbl_k.sum():>10} {avg_all:>16.4f} {avg_anom:>17.4f} {sc_k.max():>10.4f}')

blank()
print(f'  Chosen k = {k_a}  (sqrt({n_a}) = {np.sqrt(n_a):.1f})')
blank()

# ── Visualization ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('k Sensitivity Analysis — Effect of k on Anomaly Detection (702 Pitches)',
             fontsize=14, fontweight='bold', color='white', y=1.02)

for ax in axes:
    ax.set_facecolor('#1A1D27')

# Anomaly count vs k
n_anom_per_k = [k_results_a[kv]['labels'].sum() for kv in k_values_a]
axes[0].plot(k_values_a, n_anom_per_k, 'o-', color=SCORE_COLOR,
             linewidth=2.5, markersize=10, markerfacecolor='white',
             markeredgecolor=SCORE_COLOR, markeredgewidth=2.5)
axes[0].axvline(k_a, color=ACCENT, linewidth=2, linestyle='--',
                label=f'Chosen k = {k_a}')
for kv, na in zip(k_values_a, n_anom_per_k):
    axes[0].text(kv, na+0.5, str(na), ha='center', fontsize=10,
                 fontweight='bold', color='white')
axes[0].set_xlabel('k  (Number of Neighbours)', fontsize=11)
axes[0].set_ylabel('Anomalies Detected', fontsize=11)
axes[0].set_title('Anomalies Detected vs k', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, max(n_anom_per_k) * 1.4)

# Score distribution — small k vs chosen k vs large k
k_show = [k_values_a[0], k_a, k_values_a[-1]]
colors_k = [ANOMALY_COLOR, SCORE_COLOR, ACCENT]
for i, (ax, kv, ck) in enumerate(zip(axes[1:], k_show[1:], colors_k[1:])):
    sc_show = k_results_a[kv]['scores']
    lb_show = k_results_a[kv]['labels']
    ax.hist(sc_show[lb_show==0], bins=35, color=NORMAL_COLOR, alpha=0.70,
            label='Normal', edgecolor='none', density=True)
    ax.hist(sc_show[lb_show==1], bins=12, color=ANOMALY_COLOR, alpha=0.85,
            label='Anomaly', edgecolor='none', density=True)
    thr_show = np.percentile(sc_show, (1-contamination_a)*100)
    ax.axvline(thr_show, color=SCORE_COLOR, linewidth=2.5, linestyle='--',
               label=f'Threshold = {thr_show:.3f}')
    ax.set_title(f'Score Distribution  (k = {kv})', fontweight='bold')
    ax.set_xlabel('Anomaly Score'); ax.set_ylabel('Density')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Cell 11 — Anomaly Deep-Dive: What Makes These Pitches Unusual?
---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Deep-dive: why are the top anomalies flagged?
# ══════════════════════════════════════════════════════════════════════════

hdr('ANOMALY DEEP-DIVE  —  Experiment A  (Top Anomalous Pitches)')
blank()

# Get top anomalies and their feature values vs dataset median
top_anomalies_a = df_a[df_a['is_anomaly']==1].nlargest(10, 'anomaly_score')
medians_a = df_a[FEAT_A].median()

print(f'  Dataset Medians (for reference):')
sep()
for f in FEAT_A:
    print(f'  {f:<32}  {medians_a[f]:>12.2f}')
blank()

print(f'  TOP 10 ANOMALOUS PITCHES — Feature Comparison vs Median')
sep()
for rank, (_, row_) in enumerate(top_anomalies_a.iterrows(), 1):
    print(f'  Rank {rank} | {row_["Startup Name"]}  |  Industry: {row_["Industry"]}  |  Score: {row_["anomaly_score"]:.4f}')
    for f in FEAT_A:
        val    = row_[f]
        med    = medians_a[f]
        ratio  = val / med if med != 0 else float('inf')
        flag   = '  <-- EXTREME' if abs(ratio - 1) > 3 else ''
        print(f'       {f:<32}  val={val:>10.2f}  median={med:>8.2f}  ratio={ratio:>6.2f}x{flag}')
    print()

blank()

# Industry anomaly rate
hdr('ANOMALY RATE BY INDUSTRY  —  Experiment A')
ind_stats = df_a.groupby('Industry').agg(
    Total    = ('is_anomaly', 'count'),
    Anomalies= ('is_anomaly', 'sum')
).reset_index()
ind_stats['Rate%'] = (ind_stats['Anomalies'] / ind_stats['Total'] * 100).round(1)
ind_stats = ind_stats[ind_stats['Total'] >= 10].sort_values('Rate%', ascending=False)
blank()
print(f'  {"Industry":<30} {"Total":>7} {"Anomalies":>10} {"Rate%":>8}')
sep()
for _, r in ind_stats.iterrows():
    bar = '|' * int(r['Rate%'])
    print(f'  {r["Industry"]:<30} {r["Total"]:>7} {r["Anomalies"]:>10} {r["Rate%"]:>7}%  {bar}')
blank()

hdr('ANOMALY RATE BY SEASON  —  Experiment A')
season_stats = df_a.groupby('Season Number').agg(
    Total    = ('is_anomaly', 'count'),
    Anomalies= ('is_anomaly', 'sum')
).reset_index()
season_stats['Rate%'] = (season_stats['Anomalies'] / season_stats['Total'] * 100).round(1)
blank()
print(f'  {"Season":<10} {"Total Pitches":>14} {"Anomalies":>10} {"Rate%":>8}')
sep()
for _, r in season_stats.iterrows():
    print(f'  Season {int(r["Season Number"]):<4} {r["Total"]:>14} {r["Anomalies"]:>10} {r["Rate%"]:>7}%')
blank()


## Cell 12 — Final Summary
---

In [ ]:
hdr('FINAL SUMMARY  —  KNN Anomaly Detection on Shark Tank India')
blank()
print('  EXPERIMENTS COMPLETED')
sep()
print(f'  Experiment A  :  {n_a} pitches  ->  {labels_a.sum()} anomalies  ({labels_a.sum()/n_a*100:.1f}%)')
print(f'  Experiment B  :  {n_b} deals    ->  {labels_b.sum()} anomalies  ({labels_b.sum()/n_b*100:.1f}%)')
print(f'  k used (A)    :  {k_a}  (sqrt(702))')
print(f'  k used (B)    :  {k_b}  (sqrt({n_b}))')
blank()
print('  WHAT THE ANOMALIES REPRESENT')
sep()
print('  Experiment A — Anomalous Pitches:')
print('    These are startups whose combination of Ask Amount,')
print('    Offered Equity, and Valuation is very different from')
print('    all other pitches. E.g., asking Rs 30,000L for 0.5%')
print('    equity implies a Rs 60,00,000L valuation — far outside')
print('    the typical Shark Tank range.')
blank()
print('  Experiment B — Anomalous Deals:')
print('    Closed deals where the deal structure is highly unusual:')
print('    extreme deal amounts, very low/high equity for the amount,')
print('    or valuation ratios far outside the norm.')
blank()
print('  BUSINESS INSIGHTS')
sep()
print('  [1] Most anomalous pitches appear in Food & Beverage and')
print('      Technology industries — where ask amounts vary widely')
print('  [2] Early seasons (S1, S2) have slightly higher anomaly rates')
print('      as the ecosystem was less calibrated')
print('  [3] Anomalous deals tend to involve very high amounts or')
print('      unusual equity structures (e.g., royalty-based deals)')
blank()
print('  ALGORITHM CHOICES')
sep()
print('  Method    : KNN (k-Nearest Neighbours, largest distance)')
print('  Metric    : Euclidean distance in standardised feature space')
print('  Threshold : 95th percentile of anomaly scores (5% contamination)')
print('  Library   : PyOD (Python Outlier Detection)')
blank()
print('  PROS ON THIS DATASET')
sep()
print('  [+] No labels needed — purely unsupervised')
print('  [+] Works well with mixed numeric business metrics')
print('  [+] Easily interpretable: "this pitch is far from all others"')
print('  [+] n=702 is well within KNN computational limits')
blank()
print('  LIMITATIONS')
sep()
print('  [-] Cannot use categorical features (Industry, City) directly')
print('  [-] Missing values had to be imputed before fitting')
print('  [-] Contamination rate (5%) is assumed, not data-driven')
print('  [-] Features like Yearly Revenue have 50% missingness')
blank()
print('All experiments complete.')
